<a href="https://colab.research.google.com/github/HowardChen0211/dataManPy/blob/main/ProblemSet_5_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# California Housing Market, 2020-2025

## Abstract

This project looks at how home values changed in California's 58 counties from 2020 to 2025. In these six years prices went up very fast while the 30-year mortgage rate was around 3%, and then slowed down after the rate went above 5% in 2022. My main question is whether county prices mostly moved together with the mortgage rate, or whether local conditions decided where prices rose the most.

I use monthly Zillow data for home values and rents, the weekly 30-year mortgage rate from Freddie Mac, and county data from the Census Bureau.

Home values went up in 55 of 58 counties (median +31.9%), even though California lost population and 37 counties got smaller. The timing was almost the same everywhere. The month alone explains about 83% of the variation in county year-over-year growth, and which county it is explains only about 6%. The typical county grew about 11% a year while the rate was below 5% and under 2% a year after that. The size of the increase was very different across counties, from 17% on average in the North Coast to 48% in Southern California. Income and unemployment are related to how expensive a county is, but not to how fast it grew. The Bay Area is the most expensive region and it was one of the slowest growers. San Francisco ended lower than where it started.

So, if the ups and downs mostly follow a national interest rate, then local housing policy can probably change how expensive a county is in the long run, but not the swings over a few years.

## Research questions

1. Which counties had the fastest home-value growth? I expected most counties to go up, with the expensive coastal metros (especially the Bay Area) growing fastest.
2. Did rents grow along with home values? I expected yes.
3. How did home-value growth change when the mortgage rate went from below 5% to above 5%? I expected growth to slow down, with a delay of some months.
4. Where are homes least affordable compared with income? I expected the high-income coastal counties.
5. Did counties that gained people or had better job markets see faster growth? I expected yes.

## Why these data

Zillow's ZHVI and ZORI are monthly and go down to the county level, which the Census data cannot do (ACS is yearly or 5-year). ZHVI covers all 58 California counties, ZORI covers 49. The Freddie Mac 30-year fixed rate is the usual measure of what home buyers pay to borrow. The Census data (through Social Explorer) give population, unemployment, income and house value for every county, so I can test the local explanations. The unit of analysis is the county (n = 58).

## Import Libraries

In [1]:
import datetime
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pandas_datareader as pdr
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# display all output in a cell, not just the last command
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# turn off scientific notation: https://medium.com/@amit25173/steps-to-pandas-turn-off-scientific-notation-62c44b86ee24
pd.set_option('display.float_format', '{:.3f}'.format)

# ProblemSet 0 (Data)

## Load the data

In [2]:
# Zillow Home Value Index (ZHVI)
zhvi = pd.read_csv("https://github.com/HowardChen0211/dataManPy/raw/refs/heads/main/data/County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv")

# Zillow Observed Rent Index (ZORI)
zori = pd.read_csv("https://github.com/HowardChen0211/dataManPy/raw/refs/heads/main/data/County_zori_uc_sfrcondomfr_sm_month.csv")

# County population 2025
pop_2025 = pd.read_csv("https://github.com/HowardChen0211/dataManPy/raw/refs/heads/main/data/CA_Population_estimates_2025.csv")

# ACS 2020-2024 5-year: civilian labor force 16 and over
unemployment = pd.read_csv("https://github.com/HowardChen0211/dataManPy/raw/refs/heads/main/data/CA_UnemploymentRate.csv")

In [3]:
# take a look about the data (first 5 rows)
zhvi.head()

,RegionID,SizeRank,RegionName,RegionType,StateName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,2000-01-31,...,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31,2026-04-30,2026-05-31,2026-06-30,2026-07-31,2026-08-31
0,3101,0,Los Angeles County,county,CA,CA,"Los Angeles-Long Beach-Anaheim, CA",6,37,210217.606,...,876483.587,880007.554,881688.904,881664.430,879853.110,877155.644,874790.007,872663.237,872220.754,873008.422
1,139,1,Cook County,county,IL,IL,"Chicago-Naperville-Elgin, IL-IN-WI",17,31,152726.258,...,327258.568,328895.004,330363.857,332095.628,333712.156,335053.112,336017.428,337263.419,339162.650,341569.751
2,1090,2,Harris County,county,TX,TX,"Houston-The Woodlands-Sugar Land, TX",48,201,113021.492,...,283353.950,283400.843,283252.913,282917.698,282310.099,281596.697,280609.690,279554.676,278797.692,278517.967
3,2402,3,Maricopa County,county,AZ,AZ,"Phoenix-Mesa-Chandler, AZ",4,13,146106.504,...,459582.164,460367.020,461220.421,461894.031,461740.422,460735.658,459147.558,457589.894,456733.110,456252.051
4,2841,4,San Diego County,county,CA,CA,"San Diego-Chula Vista-Carlsbad, CA",6,73,216416.160,...,925424.341,927598.249,928828.819,929848.898,930228.330,930115.693,929543.266,928860.114,929330.518,930799.655


In [4]:
# take a look about the data (first 5 rows)
zori.head()

,RegionID,SizeRank,RegionName,RegionType,StateName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,2015-01-31,...,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31,2026-04-30,2026-05-31,2026-06-30,2026-07-31,2026-08-31
0,3101,0,Los Angeles County,county,CA,CA,"Los Angeles-Long Beach-Anaheim, CA",6,37,1691.320,...,2777.262,2768.420,2770.406,2776.600,2788.838,2797.135,2805.344,2809.822,2810.309,2813.614
1,139,1,Cook County,county,IL,IL,"Chicago-Naperville-Elgin, IL-IN-WI",17,31,1413.147,...,2145.212,2143.577,2157.834,2178.380,2204.530,2226.641,2248.019,2265.658,2274.611,2272.117
2,1090,2,Harris County,county,TX,TX,"Houston-The Woodlands-Sugar Land, TX",48,201,1171.144,...,1577.797,1572.948,1571.596,1569.418,1570.920,1573.802,1580.332,1584.251,1586.397,1586.095
3,2402,3,Maricopa County,county,AZ,AZ,"Phoenix-Mesa-Chandler, AZ",4,13,905.494,...,1692.043,1689.654,1690.670,1697.466,1702.306,1706.189,1711.234,1712.111,1715.836,1716.663
4,2841,4,San Diego County,county,CA,CA,"San Diego-Chula Vista-Carlsbad, CA",6,73,1628.319,...,2917.657,2912.131,2915.513,2929.541,2941.689,2951.787,2963.444,2976.321,2984.436,2993.632


In [5]:
# take a look about the data (first 5 rows)
pop_2025.head()

,FIPS Code,Summary Level,Area Name,Qualified Area Name,"Total Population (July 1, 2025)"
0,Geo_FIPS,_geo_level_,Geo_NAME,Geo_qname,ORG_T007_001
1,06001,SL050,Alameda County,"Alameda County, California",1636630
2,06003,SL050,Alpine County,"Alpine County, California",1043
3,06005,SL050,Amador County,"Amador County, California",41876
4,06007,SL050,Butte County,"Butte County, California",209211


In [6]:
# take a look about the data (first 5 rows)
unemployment.head()

,FIPS,GeoLevel,NAME,Qualified Area Name,Area (Land),Area (Water),Civilian Population in Labor Force 16 Years and Over,Civilian Population in Labor Force 16 Years and Over: Employed,Civilian Population in Labor Force 16 Years and Over: Unemployed
0,Geo_FIPS,_geo_level_,Geo_NAME,Geo_qname,Geo_AREALAND,Geo_AREAWATER,SE_A17005_001,SE_A17005_002,SE_A17005_003
1,06001,SL050,"Alameda County, California","Alameda County, California",1910010500,216909647,908995,860129,48866
2,06003,SL050,"Alpine County, California","Alpine County, California",1912292607,12557304,837,800,37
3,06005,SL050,"Amador County, California","Amador County, California",1539967082,29437117,17257,16325,932
4,06007,SL050,"Butte County, California","Butte County, California",4238545149,105204062,100743,93228,7515


**Interpretation:** Both Zillow files are national and wide, with one row per U.S. county and one column per month, so I have to filter to California and reshape them before doing anything else.

# ProblemSet 1 (Cleaning and Merging)

## Data Inspection

In [7]:
# prints information about a DataFrame
print(zhvi.info(), "\n")
print(zori.info(), "\n")
print(pop_2025.info(), "\n")
print(unemployment.info(), "\n")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3071 entries, 0 to 3070
Columns: 329 entries, RegionID to 2026-08-31
dtypes: float64(320), int64(4), object(5)
memory usage: 7.7+ MB
None 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1407 entries, 0 to 1406
Columns: 149 entries, RegionID to 2026-08-31
dtypes: float64(140), int64(4), object(5)
memory usage: 1.6+ MB
None 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59 entries, 0 to 58
Data columns (total 5 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   FIPS Code                        59 non-null     object
 1   Summary Level                    59 non-null     object
 2   Area Name                        59 non-null     object
 3   Qualified Area Name              59 non-null     object
 4   Total Population (July 1, 2025)  59 non-null     object
dtypes: object(5)
memory usage: 2.4+ KB
None 

<class 'pandas.core.frame.DataFrame'>
RangeI

In [8]:
# See how big the data
print(zhvi.shape) # (3071, 329)
print(zori.shape) # (1407, 149)
print(pop_2025.shape) # (59, 5)
print(unemployment.shape) # (59, 9)

(3071, 329)
(1407, 149)
(59, 5)
(59, 9)


In [9]:
# Missing values
print(zhvi.isna().sum().sum())
print(zori.isna().sum().sum())
print(pop_2025.isna().sum().sum())
print(unemployment.isna().sum().sum())

275007
102547
0
0


In [10]:
# Check Duplicates
print(zhvi.duplicated().sum())
print(zori.duplicated().sum())
print(pop_2025.duplicated().sum())
print(unemployment.duplicated().sum())

0
0
0
0


**Interpretation:** The population and unemployment files have 59 rows, which is the 58 counties plus one code row at the top. Both Zillow files have many missing values, mostly because some counties start later than others. None of the four files has duplicate rows.

## Data Cleaning

I reshape both Zillow files (ZHVI and ZORI) from wide to long format and keep only the months from January 2020 to December 2025.

In [11]:
START, END = "2020-01", "2026-01"

# AI (Claude): first version of this function. I asked for a function that filters a Zillow
# county file to California and reshapes it from wide to long. I cut the columns down to the
# ones I use and changed the date filter.
def zillow_to_long(data, value_name, start=START, end=END):

    cols = ["RegionID", "SizeRank", "RegionName", "RegionType", "StateName", "StateCodeFIPS", "MunicipalCodeFIPS"]
    dates = [i for i in data.columns if i[0].isdigit() and start < i < end]

    ca = data.loc[data["StateName"] == "CA", cols + dates].copy()
    # FIPS codes is 00+000
    ca["FIPS"] = (ca["StateCodeFIPS"].astype(str).str.zfill(2) + ca["MunicipalCodeFIPS"].astype(str).str.zfill(3))

    ca["RegionName"] = ca["RegionName"].str.strip()

    # wide to long https://pandas.pydata.org/docs/reference/api/pandas.melt.html
    df = ca.melt(id_vars=["FIPS", "RegionName"], value_vars=dates, var_name="Date", value_name=value_name)

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    return df.sort_values(["FIPS", "Date"]).reset_index(drop=True)

# Call the function
house = zillow_to_long(zhvi, "ZHVI")
rent = zillow_to_long(zori, "ZORI")

# See the shape & check unique values
print("ZHVI:", house.shape, "| counties:", house["FIPS"].nunique(), "| months:", house["Date"].nunique())
print("ZORI:", rent.shape, "| counties:", rent["FIPS"].nunique(), "| months:", rent["Date"].nunique())

ZHVI: (4176, 4) | counties: 58 | months: 72
ZORI: (3528, 4) | counties: 49 | months: 72


ZHVI covers all 58 counties × 72 months = 4,176 rows. ZORI has fewer counties because Zillow does not publish rents for some small rural counties.

## Data Manipulation

### Merge 1: ZHVI x ZORI

In [12]:
housing = house.merge(rent.drop(columns="RegionName"), on=["FIPS", "Date"],
                     how="outer", validate="1:1", indicator=True)

housing["_merge"].value_counts()

no_rent = sorted(housing.loc[housing["_merge"] == "left_only", "RegionName"].unique())
print(len(no_rent), "counties have no ZORI at all:", no_rent, "\n")

housing = housing.drop(columns="_merge")

,count
_merge,
both,3528
left_only,648
right_only,0


9 counties have no ZORI at all: ['Alpine County', 'Colusa County', 'Lassen County', 'Mariposa County', 'Modoc County', 'Mono County', 'Plumas County', 'Sierra County', 'Trinity County'] 



**Interpretation:** I used an outer join to see what does not match. All 4,176 county-months are kept. The 648 `left_only` rows are 9 small rural counties (9 x 72) where Zillow does not publish a rent index, probably because there are too few rental listings. This is expected and not a merge problem.

### Merge 2: Unemployment x Population

#### Population (from Social Explorer)

In [13]:
pop_2025 = (pop_2025.iloc[1:,:][["FIPS Code", "Area Name", "Total Population (July 1, 2025)"]].rename(columns={"FIPS Code": "FIPS", "Area Name": "County", "Total Population (July 1, 2025)": "pop_2025"}))

pop_2025["pop_2025"] = pop_2025["pop_2025"].astype(int)

pop_2025.shape
pop_2025.head()

(58, 3)

,FIPS,County,pop_2025
1,06001,Alameda County,1636630
2,06003,Alpine County,1043
3,06005,Amador County,41876
4,06007,Butte County,209211
5,06009,Calaveras County,46605


#### Unemployment (ACS 2020–2024 5-year Estimates)

In [14]:
# row 0 holds the (Geo_FIPS, SE_A17005_001, ...), so drop it
unemployment = (unemployment
                .loc[1:, ["FIPS", "NAME",
                          "Civilian Population in Labor Force 16 Years and Over",
                          "Civilian Population in Labor Force 16 Years and Over: Unemployed"]]
                .rename(columns={"NAME": "County",
                                 "Civilian Population in Labor Force 16 Years and Over": "labor_force",
                                 "Civilian Population in Labor Force 16 Years and Over: Unemployed": "unemployed"}))

unemployment["County"] = unemployment["County"].str.replace(", California", "")

unemployment["FIPS"] = unemployment["FIPS"].astype(str).str.zfill(5)

unemployment[["labor_force", "unemployed"]] = unemployment[["labor_force", "unemployed"]].astype(int)

unemployment["unemp_rate"] = unemployment["unemployed"] / unemployment["labor_force"] * 100

unemployment.shape

unemployment.head()

(58, 5)

,FIPS,County,labor_force,unemployed,unemp_rate
1,06001,Alameda County,908995,48866,5.376
2,06003,Alpine County,837,37,4.421
3,06005,Amador County,17257,932,5.401
4,06007,Butte County,100743,7515,7.460
5,06009,Calaveras County,19422,877,4.515


In [15]:
labor = unemployment.merge(pop_2025.drop(columns="County"), on="FIPS", how="outer", validate="1:1", indicator=True)

print(labor["_merge"].value_counts())

labor = labor.drop(columns="_merge")

labor.head()

_merge
both          58
left_only      0
right_only     0
Name: count, dtype: int64


,FIPS,County,labor_force,unemployed,unemp_rate,pop_2025
0,06001,Alameda County,908995,48866,5.376,1636630
1,06003,Alpine County,837,37,4.421,1043
2,06005,Amador County,17257,932,5.401,41876
3,06007,Butte County,100743,7515,7.460,209211
4,06009,Calaveras County,19422,877,4.515,46605


**Interpretation:** All 58 counties matched on both sides, with no `left_only` or `right_only` rows. That's what I expected, since both files are Social Explorer exports for the same list of California counties and FIPS is kept as a 5-digit string in both.

## Mapping: Counties to Regions

I group the counties into six regions, based on: https://coolcalifornia.arb.ca.gov/regions

In [16]:
regions = {
    "Bay Area": ["Alameda", "Contra Costa", "Marin", "Napa", "San Francisco",
                 "San Mateo", "Santa Clara", "Solano", "Sonoma"],
    "Southern California": ["Imperial", "Los Angeles", "Orange", "Riverside",
                            "San Bernardino", "San Diego", "Ventura"],
    "Central Coast": ["Monterey", "San Benito", "San Luis Obispo", "Santa Barbara", "Santa Cruz"],
    "Central Valley": ["Butte", "Colusa", "Fresno", "Glenn", "Kern", "Kings", "Madera",
                       "Merced", "Sacramento", "San Joaquin", "Stanislaus", "Sutter",
                       "Tehama", "Tulare", "Yolo", "Yuba"],
    "Sierra / Gold Country": ["Alpine", "Amador", "Calaveras", "El Dorado", "Inyo",
                              "Mariposa", "Mono", "Nevada", "Placer", "Plumas",
                              "Sierra", "Tuolumne"],
    "North Coast / Far North": ["Del Norte", "Humboldt", "Lake", "Lassen", "Mendocino",
                                "Modoc", "Shasta", "Siskiyou", "Trinity"]}

county_to_region = {f"{c} County": region for region, counties in regions.items() for c in counties}

housing["region"] = housing["RegionName"].map(county_to_region)

housing.drop_duplicates("FIPS")["region"].value_counts() # Total should be 58

,count
region,
Central Valley,16
Sierra / Gold Country,12
Bay Area,9
North Coast / Far North,9
Southern California,7
Central Coast,5


**Interpretation:** Every county got a region, since the counts add up to 58 and nothing was left. The groups are pretty uneven, the Central Valley has 16 counties and the Central Coast only 5, so the averages for the smaller regions rest on just a handful of places.

# ProblemSet 2 (Home Values and Rents)

## County-level growth (29 counties)

I first aggregate the monthly Zillow data to the county level, so I can compare how much home values and rents changed in each county from January 2020 to December 2025.

Note: For the rent comparison I keep only the counties that have ZORI data for all 72 months, so every county is compared over the same period.

In [17]:
housing_full_zori_county = housing[~housing["ZORI"].isna()]

housing_full_zori_county = housing_full_zori_county.groupby(["FIPS", "RegionName"]).agg(months=("Date", "nunique"))

# This line of code I'm using AI:Gemini to help me how to get the values from index
housing_full_zori_county = housing_full_zori_county[housing_full_zori_county["months"] == 72].index.get_level_values('RegionName').unique().to_list()

In [18]:
len(housing_full_zori_county)
housings = housing[housing["RegionName"].isin(housing_full_zori_county)].reset_index(drop=True)

29

**Interpretation:** Only 29 of the 58 counties have rent data for all 72 months, so the rent-versus-home-value comparison is based on these 29. Most of the dropped counties are small rural ones whose ZORI series starts late or doesn't exist at all.

### RQ1 (29 counties): Which counties saw the fastest home-value growth?

In [19]:
# Group monthly housing data by county
county_housing = (
    housings
    .sort_values(["FIPS", "Date"])
    .groupby(["FIPS", "RegionName", "region"])
    .agg(
        zhvi_2020=("ZHVI", "first"), # Jan 2020
        zhvi_2025=("ZHVI", "last"),  # Dec 2025
        zori_2020=("ZORI", "first"),
        zori_2025=("ZORI", "last"),
        months=("Date", "nunique")).reset_index())

# Calculate percentage growth
county_housing["zhvi_growth_pct"] = ((county_housing["zhvi_2025"] - county_housing["zhvi_2020"])
                                      / county_housing["zhvi_2020"] * 100)

county_housing["zori_growth_pct"] = ((county_housing["zori_2025"] - county_housing["zori_2020"])
                                      / county_housing["zori_2020"] * 100)

county_housing.sort_values("zhvi_growth_pct", ascending=False).reset_index(drop=True).head(10)

,FIPS,RegionName,region,zhvi_2020,zhvi_2025,zori_2020,zori_2025,months,zhvi_growth_pct,zori_growth_pct
0,06083,Santa Barbara County,Central Coast,604363.958,970185.816,2111.031,3125.583,72,60.530,48.060
1,06029,Kern County,Central Valley,236887.162,362888.534,1139.586,1730.258,72,53.190,51.832
2,06073,San Diego County,Southern California,606555.564,927598.249,2082.318,2912.131,72,52.929,39.850
3,06065,Riverside County,Southern California,402042.996,605008.058,1740.369,2554.693,72,50.483,46.790
4,06071,San Bernardino County,Southern California,369240.934,547407.809,1648.598,2412.321,72,48.252,46.326
5,06107,Tulare County,Central Valley,242725.451,358608.931,1124.410,1690.461,72,47.743,50.342
6,06019,Fresno County,Central Valley,278536.136,405709.400,1342.323,1988.009,72,45.658,48.102
7,06079,San Luis Obispo County,Central Coast,617020.411,891942.555,1955.495,2780.955,72,44.556,42.212
8,06085,Santa Clara County,Bay Area,1125886.109,1608192.584,3064.590,3540.084,72,42.838,15.516
9,06047,Merced County,Central Valley,295777.004,418573.042,1337.669,1995.053,72,41.516,49.144


**Interpretation:** Santa Barbara leads at 60.5%, followed by Kern, San Diego, and Riverside, all above 50%. What stands out is how many cheaper inland counties make the top ten (Kern, Tulare, Fresno, Merced), and in those places rents grew about as fast as home values, roughly 48-52%. Santa Clara is the odd one out: home values rose 43% but rent only 16%.

Note: This ranking only includes the 29 counties with full rent data. The full ranking of all 58 counties is in the PS3 & 4 & 5 section, where Orange County (59.8%) ranks second.

### RQ2: Did rents grow along with home values? (29 counties)

In [ ]:
print("correlation:", round(county_housing["zhvi_growth_pct"].corr(county_housing["zori_growth_pct"]), 2))

# AI (Claude): I asked how to improve this scatter. One trend line for all regions
# (trendline_scope="overall") and the hover labels came from the answer.
px.scatter(county_housing, x="zhvi_growth_pct", y="zori_growth_pct", color="region", hover_name="RegionName",
           trendline="ols", trendline_scope="overall",
           title="Home-value growth vs. rent growth, 2020-2025 (29 counties with full rent data)",
           labels={"zhvi_growth_pct": "ZHVI growth (%)", "zori_growth_pct": "ZORI growth (%)", "region": "Region"}).show()

county_housing.sort_values("zori_growth_pct")[["RegionName", "zhvi_growth_pct", "zori_growth_pct"]].head(3)

correlation: 0.72


**Interpretation:** Across the 29 counties, home-value growth and rent growth are strongly related (correlation 0.72). Counties where home values grew faster usually also had faster rent growth.

## Regional growth (all 58 counties)

Next, I aggregate the county-level growth by region to see broader patterns instead of single counties. This step uses all 58 counties, since it only needs home values.

In [ ]:
# Group monthly housing data by county
county_housing = (
    housing
    .sort_values(["FIPS", "Date"])
    .groupby(["FIPS", "RegionName"])
    .agg(
        zhvi_2020=("ZHVI", "first"),
        zhvi_2025=("ZHVI", "last"),
        months=("Date", "nunique")).reset_index())

# Calculate percentage growth
county_housing["zhvi_growth_pct"] = ((county_housing["zhvi_2025"] - county_housing["zhvi_2020"])
                                      / county_housing["zhvi_2020"] * 100)

# Add the regional classification
county_housing["region"] = county_housing["RegionName"].map(county_to_region)

regional_housing = (county_housing.groupby("region").agg(
        avg_zhvi_growth=("zhvi_growth_pct", "mean"),
        counties=("FIPS", "nunique")).reset_index().sort_values("avg_zhvi_growth", ascending=False))

regional_housing

**Interpretation:** Southern California grew the fastest (48.2% on average), followed by the Central Coast (42.0%) and the Central Valley (37.7%). The Bay Area (22.4%) and the North Coast (17.3%) grew the slowest. Some regions have only a few counties (the Central Coast has 5), so these averages should be read with caution.

# ProblemSet 3 & 4 & 5 (Mortgage Rates, Affordability, and Local Factors)


## Home values for all 58 counties

The county-level step above used `housings`, which keeps only the 29 counties with full rent data, so here I go back to all 58. Year-over-year growth needs the same month one year earlier, and `zhvi_CA` from the cleaning step starts in 2020, so I reload ZHVI from January 2019, compute YoY growth for each county, and then keep 2020-2025.

In [ ]:
zhvi_CA = zillow_to_long(zhvi, "ZHVI", "2019-01", "2026-01")

print(zhvi_CA.shape, "| counties:", zhvi_CA["FIPS"].nunique(), "| months:", zhvi_CA["Date"].nunique())

In [ ]:
home = zhvi_CA.copy()

home["region"] = home["RegionName"].map(county_to_region)
home["ym"] = home["Date"].dt.to_period("M") # month key for the merges

# year-over-year growth
home = home.sort_values(["FIPS", "Date"]).reset_index(drop=True)
home["zhvi_yoy"] = home.groupby("FIPS")["ZHVI"].pct_change(12, fill_method=None) * 100

# Keep only the original PS3 analysis period: 2020–2025.
home = home.loc[home["Date"] >= "2020-01"].reset_index(drop=True)

print(home.shape, "| counties:", home["FIPS"].nunique(), "| months:", home["ym"].nunique())
print("missing ZHVI:", home["ZHVI"].isna().sum(), "| missing YoY:", home["zhvi_yoy"].isna().sum())

# which rows have no YoY value?
home.loc[home["zhvi_yoy"].isna(), ["FIPS", "RegionName", "Date", "ZHVI", "zhvi_yoy"]]

**Interpretation:** Only four growth values are missing, and all four are Mariposa. Zillow has no home value for Mariposa in September and October 2020, so those two months have no growth, and September and October 2021 have nothing one year earlier to compare with.

## 30-Year Mortgage Rate

The 30-year fixed rate mortgage average in the United States, from Freddie Mac's weekly Primary Mortgage Market Survey.

Source: https://fred.stlouisfed.org/series/MORTGAGE30US

### Load the data

In [ ]:
start = datetime.datetime(2019, 1, 1)
end = datetime.datetime(2025, 12, 31)

# Freddie Mac PMMS 30-year fixed rate, weekly
mortgage_raw = pdr.DataReader('MORTGAGE30US', 'fred', start, end) # MORTGAGE30US is the code for series: know it from fed's url: https://fred.stlouisfed.org/series/MORTGAGE30US
mortgage_raw

In [ ]:
mortgage = (
    mortgage_raw
    .reset_index()
    .rename(columns={"DATE": "week", "MORTGAGE30US": "rate30"}))

mortgage["week"] = pd.to_datetime(mortgage["week"])
mortgage["rate30"] = pd.to_numeric(mortgage["rate30"], errors="coerce")

print(mortgage.shape, "| missing rates:", mortgage["rate30"].isna().sum())
print("duplicate weeks:", mortgage["week"].duplicated().sum())

# PMMS is released on Thursdays, except some holiday weeks
mortgage["week"].dt.day_name().value_counts()

**Interpretation:** 353 of the 366 weekly observations fall on a Thursday and the other 13 on a Wednesday. Those are holiday weeks when Freddie Mac released a day early, which is one more reason not to match on the exact date.

### Clean the data: weekly to monthly

Zillow is monthly, so I average the 4-5 weekly observations in each month. I also build two variables: the 12-month change in the rate, and a recode into a low-rate (<5%) vs. high-rate (5%+) regime.

In [ ]:
mortgage_m = (mortgage
              .assign(mth = mortgage["week"].dt.to_period("M"))
              .groupby("mth", as_index=False)
              .agg(rate30=("rate30", "mean"),
                   n_weeks=("rate30", "count")))

# change vs. same month last year, in percentage points
mortgage_m["rate30_chg12"] = mortgage_m["rate30"] - mortgage_m["rate30"].shift(12)

# regime recode
mortgage_m["rate_regime"] = "Low rate (<5%)"
mortgage_m.loc[mortgage_m["rate30"] >= 5, "rate_regime"] = "High rate (5%+)"

print(mortgage_m.shape) # expect 84 months = 7 years x 12

# first and last month of each regime
mortgage_m.groupby("rate_regime").agg(months=("mth", "count"), first_month=("mth", "min"), last_month=("mth", "max"))

**Interpretation:** There are 84 months because I kept 2019 for the 12-month change. That change shows how fast borrowing got more expensive: in October 2022 the rate was 3.8 points higher than a year earlier. The rate was under 5% for 40 months in a row, from January 2019 to April 2022, and it stayed at 5% or more for all 44 months from May 2022 to December 2025. So the recode splits the period cleanly into a before and an after, with no going back and forth.

In [ ]:
home_rate = home.merge(
    mortgage_m,
    left_on="ym",
    right_on="mth",
    how="outer",          # outer on purpose to see what doesn't match
    validate="m:1",       # many county-months to one national rate per month
    indicator=True)

home_rate["_merge"].value_counts()

# crosstab the merge result
pd.crosstab(home_rate["mth"].dt.year, home_rate["_merge"])

**Interpretation:** Every county-month found a rate (696 per year = 58 x 12), so nothing on the housing side is lost. The 12 `right_only` rows are the 2019 months I kept only for the 12-month change. The Zillow data starts in 2020, so they're expected.

In [ ]:
# keep the matched rows
home_rate = (home_rate[home_rate["_merge"] == "both"]
             .drop(columns=["_merge", "n_weeks"])
             .reset_index(drop=True))

home_rate.head(10)

### RQ3: How did home-value growth change when the 30-year mortgage rate went from below 5% to above 5%?

In [ ]:
state_month = (home_rate.groupby("Date", as_index=False).agg(zhvi_yoy=("zhvi_yoy", "median"),
                                                             rate30=("rate30", "first")))

# AI (Claude): I asked how to put two lines with different scales on one plotly chart.
fig = (make_subplots(specs=[[{"secondary_y": True}]])
       .add_trace(go.Scatter(x=state_month["Date"], y=state_month["zhvi_yoy"],
                             name="Home value YoY % (median of 58 counties)"), secondary_y=False)
       .add_trace(go.Scatter(x=state_month["Date"], y=state_month["rate30"],
                             name="30-yr mortgage rate %", line=dict(dash="dot", color="black")),
                  secondary_y=True)
       .add_hline(y=0, line_width=1, line_color="gray")
       .update_layout(title="California home-value growth vs. the 30-year mortgage rate, 2020-2025",
                      hovermode="x unified", legend=dict(orientation="h", y=-0.2))
       .update_yaxes(title_text="YoY growth (%)", secondary_y=False)
       .update_yaxes(title_text="Mortgage rate (%)", secondary_y=True))
##

fig.show()

print("\n Correlation between the two lines:", round(state_month["zhvi_yoy"].corr(state_month["rate30"]), 2))

**Interpretation:** Home-value growth and the mortgage rate moved in opposite directions (correlation -0.69). Growth peaked at 21.1% in August 2021, when the rate was 2.8%, and bottomed at -5.8% in July 2023, when the rate was 6.8%. The change came with a delay, the rate passed 5% in May 2022, but the median county's did not turn negative until February 2023.

In [ ]:
home_rate.groupby("rate_regime")["zhvi_yoy"].agg(["mean", "median", "count"]).round(1)

**Interpretation:** In the 28 months when the 30-year rate was below 5% (Jan 2020 - Apr 2022), the average county's home value grew 11.2% year over year (median 10.9%). In the 44 months at 5% or above (May 2022 - Dec 2025), average growth fell to 1.7% (median 0.9%). The drop did not happen the moment rates crossed 5%: because of the delay seen in the chart above, the first few high-rate months still show double-digit growth, so the slowdown in the later high-rate months is even sharper than this average suggests.

## Affordability: ACS(American Community Survey) Data

Source: https://www.socialexplorer.com/data-library/2 2020-2024 (5-Year-Estimates)

### Load the data (Household Income, House Value)

In [ ]:
# Median household income at County level
household_income = pd.read_csv("https://raw.githubusercontent.com/HowardChen0211/gisPy/refs/heads/main/household_income_2024.csv", skiprows=[1])
household_income.rename(columns={"Median Household Income (In 2024 Inflation Adjusted Dollars)": "Median Household Income"}, inplace=True)
household_income.head()

In [ ]:
# Median owner-occupied house value at County level
house_val = pd.read_csv("https://raw.githubusercontent.com/HowardChen0211/gisPy/refs/heads/main/housevalue_2024.csv", skiprows=[1])
house_val.head()

In [ ]:
household_income.FIPS = household_income.FIPS.astype(str).str.zfill(5)
house_val.FIPS = house_val.FIPS.astype(str).str.zfill(5)

### Clean the data

In [ ]:
census_house = (
    pop_2025[["FIPS", "County", "pop_2025"]]
    .merge(household_income[["FIPS", "Median Household Income"]], on="FIPS", how="outer", validate="one_to_one")
    .merge(house_val[["FIPS", "Median House Value"]], on="FIPS", how="outer", validate="one_to_one"))

num_cols = ["pop_2025", "Median Household Income", "Median House Value"]
census_house[num_cols] = census_house[num_cols].apply(pd.to_numeric, errors="coerce") # to numeric types, if missing values found will fill it NaN

census_house.rename(columns={"pop_2025": "TPOP",
                     "Median Household Income": "MED_INC",
                     "Median House Value": "MED_HVAL"}, inplace = True)
census_house.head()

In [ ]:
house_unemp = pd.merge(census_house, unemployment, on = ["FIPS", "County"], how = "outer", validate = "1:1", indicator = True)
house_unemp["_merge"].value_counts() # expected that 58 counties match

### RQ4: Where are homes least affordable compared with income?

####  Home Price to Income Ratio

In [ ]:
house_unemp["Price2Income"] = house_unemp["MED_HVAL"] / house_unemp["MED_INC"]  # Home Price to Income Ratio

# least affordable counties
house_unemp.sort_values("Price2Income", ascending=False)[["County", "MED_INC", "MED_HVAL", "Price2Income"]].reset_index(drop=True).head(10)

#### Scatter plots 0

In [ ]:
fig = px.scatter(house_unemp, x="MED_INC", y="MED_HVAL",
                 color="Price2Income", hover_name="County",
                 color_continuous_scale="YlOrRd", labels={"Price2Income": "P/I Ratio"})

## Done with AI (Claude)
fig = fig.update_layout(
    title_text="Home Value vs. Household Income by County, California (ACS 2020–2024 5-Year Estimates)",
    xaxis_title="Median Household Income",
    yaxis_title="Median Home Value",
    coloraxis_colorbar=dict(title="P/I Ratio"))
##

fig.show()

**Interpretation:** Counties with higher incomes also have more expensive homes, but home values go up more steeply than incomes across counties. In lower-income counties a typical home costs about 4 to 6 years of income, while in the highest-income counties it costs about 9 to 10 years (Marin 10.1, San Francisco 9.9, San Mateo 9.8). So the counties where people earn the most are the hardest places to buy a home. Los Angeles is the clearest exception, and its median income is only about \$90k, but homes cost over \$800k, which puts it fourth on the list.

In [ ]:
# add the region to each county
house_unemp["region"] = house_unemp["County"].map(county_to_region)


# price-to-income by region
(house_unemp.groupby("region")
     .agg(counties=("FIPS", "nunique"),
          median_income=("MED_INC", "median"),
          median_house_value=("MED_HVAL", "median"),
          median_p2i=("Price2Income", "median"))
     .reset_index()
     .sort_values("median_p2i", ascending=False))

**Interpretation:** To see if location matters, I grouped California's counties into six regions and took the median of each region's counties. The three coastal regions, the Bay Area, Central Coast, and Southern California, all have a price-to-income ratio of about 7.5 or higher. The three inland regions, Sierra / Gold Country, Central Valley, and North Coast / Far North, are all between 5.2 and 5.6. There is almost nothing in the middle, so the biggest divide is coast versus inland. For example, the Bay Area, where it has the highest income at about \$129k, but homes cost over \$1 million, so it is still the least affordable region. On the other side, the inland regions have lower incomes, but homes are much cheaper, so housing is easier to afford.

## RQ5: Did counties that gained people or had stronger job markets see faster home-value growth?

I look at two local factors: the job market and population change.

### Part 1: Job market (unemployment rate)

#### Merge: ZHVI x Unemployment

In [ ]:
# monthly ZHVI, 2020-01 to 2025-12
county_zhvi = (
    home.sort_values(["FIPS", "Date"]).groupby(["FIPS", "RegionName", "region"])
    .agg(zhvi_2020=("ZHVI", "first"),    # Jan 2020
         zhvi_2025=("ZHVI", "last"),     # Dec 2025
         months=("Date", "nunique"))
    .reset_index())

county_zhvi["zhvi_growth_pct"] = (county_zhvi["zhvi_2025"] / county_zhvi["zhvi_2020"] - 1) * 100

county_zhvi.sort_values("zhvi_growth_pct", ascending=False).reset_index(drop=True).head()

In [ ]:
growth = county_zhvi.merge(
    unemployment.drop(columns="County"),
    on="FIPS",
    how="outer",
    validate="1:1",
    indicator=True)

growth["_merge"].value_counts() # It's expected.

#### Scatter plots 1

In [ ]:
df = growth.dropna(subset=["unemp_rate", "zhvi_2025", "zhvi_growth_pct"]).copy()
df["log_zhvi_2025"] = np.log(df["zhvi_2025"])

for col in ["zhvi_2025", "log_zhvi_2025", "zhvi_growth_pct"]:
    p = df["unemp_rate"].corr(df[col])
    print(f"{col:16s} pearson= {p:.3f}")

fig = px.scatter(
    df, x="unemp_rate", y="zhvi_2025", color="region",
    hover_name="RegionName", log_y=True,
    trendline="ols", trendline_scope="overall",
    trendline_options=dict(log_y=True),
    title="Unemployment Rate vs. Home Value (Dec 2025), California counties",
    labels={"unemp_rate": "Unemployment Rate (%)",
            "zhvi_2025": "ZHVI, Dec 2025 (log scale)"})
fig.show()

**Interpretation:** The chart shows where unemployment does matter: counties with higher unemployment have cheaper homes (correlation -0.51, or -0.56 on a log scale). The local job market helps explain which counties are expensive, not which ones grew the fastest.

The chart uses a log scale because home values are very skewed. Most counties are between 300 and 600 thousand dollars, but a few Bay Area counties are over 1 million. The log keeps those few counties from driving the result, and it compares counties by how many times more expensive they are, not by the dollar gap.

### Part 2: Population

Source: Census Bureau; Social Explorer. Total Population, 2020. Prepared by Social Explorer. https://www.socialexplorer.com/data/CDHC2020/metadata?ds=SE&table=T001

#### Load the data

In [ ]:
pop_2020 = pd.read_csv("https://github.com/HowardChen0211/dataManPy/raw/refs/heads/main/data/TPOP_2020.csv")
pop_2020.head()

#### Clean the data

In [ ]:
pop_2020["Geo_FIPS"] = pop_2020["Geo_FIPS"].astype(str).str.zfill(5)

pop_2020.rename(columns={"Geo_FIPS":"FIPS", "Geo_NAME":"County", "SE_T001_001": "pop_2020"}, inplace=True)

pop_2020 = pop_2020[["FIPS", "County", "pop_2020"]]

pop_2020.head(10)

##### Merge: Population 2020 x Population 2025

In [ ]:
pop_growth = pop_2020.merge(
    pop_2025.drop(columns="County"),
    on="FIPS",
    how="outer",          # outer on purpose, to see what doesn't match
    validate="1:1",
    indicator=True)

pop_growth["_merge"].value_counts()

##### Population growth, 2020 - 2025


In [ ]:
pop_growth = pop_growth.drop(columns="_merge")

# population change, in people and in percent
pop_growth["pop_change"] = pop_growth["pop_2025"] - pop_growth["pop_2020"]
pop_growth["pop_growth_pct"] = (pop_growth["pop_2025"] / pop_growth["pop_2020"] - 1) * 100

# statewide total (AI Done by Claude)
ca_2020, ca_2025 = pop_growth["pop_2020"].sum(), pop_growth["pop_2025"].sum()
print(f"California: {ca_2020:,} to {ca_2025:,} ({(ca_2025 / ca_2020 - 1) * 100:.2f}%)\n")
print("Counties that lost population:", (pop_growth["pop_growth_pct"] < 0).sum(), "of", len(pop_growth))

# five fastest-growing and five fastest-shrinking counties
pop_sorted = pop_growth.sort_values("pop_growth_pct", ascending=False).reset_index(drop=True)
pop_sorted.head()
print()
pop_sorted.tail()

**Interpretation:** California lost about 0.5% of its population between 2020 and 2025, and 37 of the 58 counties shrank. Placer, San Benito, and Yuba grew the fastest (about 9%), while Lassen and Alpine lost more than 13%. Note that the 2020 number is the Census count and the 2025 number is an estimate, so small changes should be read with caution.

#### Home-value growth, 2020-2025 (all 58 counties)

In [ ]:
print("median growth:", round(county_zhvi["zhvi_growth_pct"].median(), 1))

print("counties with higher home values:", (county_zhvi["zhvi_growth_pct"] > 0).sum(), "of", len(county_zhvi))

ranked = county_zhvi.sort_values("zhvi_growth_pct", ascending=False)[["RegionName", "region", "zhvi_growth_pct"]]
ranked.head(10)
ranked.tail(3)

**Interpretation:** With all 58 counties, Santa Barbara (60.5%) and Orange (59.8%) lead, and Southern California and the Central Valley take eight of the top ten. The median county grew 31.9%, and only Mendocino (-4.8%), Trinity (-3.1%), and San Francisco (-2.4%) ended lower than they started. This is the full answer to RQ1.

##### Merge: ZHVI x Population Growth

In [ ]:
growth = county_zhvi.merge(
    pop_growth.drop(columns="County"),
    on="FIPS",
    how="outer",
    validate="1:1",
    indicator=True)

growth["_merge"].value_counts() # It's expected.

#### Scatter plots 2

In [ ]:
growth = growth.drop(columns="_merge")

print("correlation:", round(growth["pop_growth_pct"].corr(growth["zhvi_growth_pct"]), 3))

shrinking = growth[growth["pop_growth_pct"] < 0]
print(f"{(shrinking['zhvi_growth_pct'] > 0).sum()} of {len(shrinking)} shrinking counties still saw home values rise")

fig = px.scatter(growth, x="pop_growth_pct", y="zhvi_growth_pct", color="region",
                 hover_name="RegionName", trendline="ols", trendline_scope="overall",
                 title="Population Growth vs. home-value Growth, California counties, 2020-2025",
                 labels={"pop_growth_pct": "Population Growth (%)",
                         "zhvi_growth_pct": "Home-value Growth (%)"})
fig.show()

**Interpretation:** The line goes up a little, so counties that grew faster tended to see slightly bigger price increases, but the link is weak (correlation 0.26) and the dots are spread widely around the line. Home values went up almost everywhere, even where people were leaving: 34 of the 37 counties that lost population still saw home values rise. Orange County, for example, lost about 1% of its population, but its home values went up almost 60%. So population change explains little of the price increase, and other factors probably mattered more. This is also only a correlation, so it cannot tell us whether population growth causes prices to go up.

# Conclusion

Going back to the five questions:

1. Most counties went up (55 of 58), which I expected. But the fastest growth was in Southern California and the Central Valley, and the Bay Area was the second slowest region. So my guess was half right.
2. Yes. Rent growth and home-value growth have a correlation of 0.72 in the 29 counties with full data.
3. Yes. Average growth fell from 11.2% to 1.7% a year after the rate passed 5%, and the effect was strongest about 7 to 8 months later.
4. Mostly yes. Marin, San Francisco and San Mateo cost about 10 years of income. Los Angeles is fourth with a much lower income, so the coast matters more than income.
5. Mostly no. Population growth has a weak relation with price growth (0.26) and unemployment has none (0.02). They are related to the price level, not to growth.

Between 2020 and 2025 home prices in California rose and slowed at the same time everywhere, and the timing follows the mortgage rate. The month explains 83% of the variation in growth and the county only 6%. How much each county gained in total is a different story and I could not explain it well with population, unemployment or income. What surprised me most was the Bay Area. It is the most expensive region, but it grew slowly, its rents hardly moved, and San Francisco ended below where it started. I think remote work could be part of it, but I do not have data on that in this project.

# Limitations and future research

Limitations:

- Everything here is a correlation. The mortgage rate and home prices could both be reacting to something else, such as inflation or the economy after COVID. There is also only one rate cycle in the period, so in a way I have one observation of "rates go up".
- Prices are not adjusted for inflation. Consumer prices also rose a lot in these years, so the real increase is smaller than 31.9%.
- The time periods do not match exactly. Zillow growth is January 2020 to December 2025, the ACS numbers are 2020-2024 averages, and population compares the April 2020 Census with a July 2025 estimate.
- The ACS unemployment rate is a 5-year average that includes the COVID spike in 2020, and it is measured during the same years as the price growth, not before.
- The samples are small: 58 counties, 29 for rents, and only 5 in the Central Coast. A few unusual counties can move a result.
- Every county counts the same, so Alpine (about 1,000 people) weighs as much as Los Angeles (about 9.7 million).
- Home-value growth uses Zillow, but the price-to-income ratio uses the ACS house value, which is what owners say their home is worth. The two are close but not the same.
- Year-over-year growth is slow to react, so turning points look later than they really were.

Future work (I may continue this project next year):

- Adjust prices for inflation with CPI to get real growth.
- Use month-over-month growth and a regression with lags, with county and month effects, to measure the delay better than a simple correlation.
- Add housing supply (building permits, for-sale inventory) and remote-work or migration data to explain why the Bay Area was different.
- Weight counties by population, or go down to ZIP codes, to see what most people experienced.
- Extend the period back before 2020 or add other states, so there is more than one rate cycle to compare.

# References

1. Zillow Home Value Index, county, all homes, mid tier, smoothed, seasonally adjusted, monthly: https://files.zillowstatic.com/research/public_csvs/zhvi/County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv

2. Zillow Observed Rent Index, county, all homes plus multifamily, smoothed, monthly: https://files.zillowstatic.com/research/public_csvs/zori/County_zori_uc_sfrcondomfr_sm_month.csv

3. Freddie Mac, 30-Year Fixed Rate Mortgage Average in the United States (MORTGAGE30US), weekly, from FRED: https://fred.stlouisfed.org/series/MORTGAGE30US

4. U.S. Census Bureau; Social Explorer. Total Population, 2020 Census: https://www.socialexplorer.com/data/CDHC2020/metadata?ds=SE&table=T001

5. U.S. Census Bureau, Population Estimates Program, county population totals, July 1, 2025: https://www.census.gov/data/tables/time-series/demo/popest/2020s-counties-total.html

6. U.S. Census Bureau; Social Explorer. ACS 2020-2024 5-Year Estimates, Employment Status for Total Population 16 Years and Over: https://www.socialexplorer.com/data/ACS2024_5yr/metadata?ds=SE&table=A17005

7. U.S. Census Bureau; Social Explorer. ACS 2020-2024 5-Year Estimates, Median Household Income: https://www.socialexplorer.com/data/ACS2024_5yr/metadata?ds=SE&table=A14006

8. U.S. Census Bureau; Social Explorer. ACS 2020-2024 5-Year Estimates, Median House Value: https://www.socialexplorer.com/data/ACS2024_5yr/metadata?ds=SE&table=A10036

9. California Air Resources Board, Cool California regions: https://coolcalifornia.arb.ca.gov/regions

## Use of AI

I used Claude (https://claude.ai) in this project. Code where the first version or the idea came from Claude is marked with a comment that starts with `# AI (Claude):`, and the comment says what I asked and what I changed. I also asked Claude to review the whole notebook before submitting. The prompt was roughly: "look at my notebook, tell me what can be tightened or improved in the code, analysis and charts, and check the numbers in my text against the output".